# Introduction 

<font size="3">This notebook demonstrates the I2IwFiLM translation pipeline from white-light imagery to CaIIK imagery, starting from an active-region-centered crop of a white-light solar image in .npz format. This will allow you to understand the translation pipeline architecture, learn how to load pretrained models and run inference on new solar observations.

<div style="width: 750px;  padding: 15px; border: 1px solid transparent; border-color: transparent; margin-bottom: 20px; border-radius: 4px; color: #8a6d3b;; background-color: #fcf8e3; border-color: #faebcc;">
This tutorial requires pretrained translation model from the I2IwFiLM package. Model training is covered separately in the repository's README.md. Ensure you have completed the two-stages training or obtained pretrained weights before proceeding.
</div>

# Code

<font size="3">Import required libraries and dependencies for the pipeline.</font>

In [ ]:
import os
import sys
import glob
from pathlib import Path 
sys.path.append('..')

import json
from omegaconf import OmegaConf
from copy import deepcopy

from tqdm.notebook import tqdm
from datetime import datetime
import matplotlib
import matplotlib.pyplot as plt
import ipywidgets as widgets

from hydra.utils import instantiate
from i2iwfilm.load import load_from_dir_without_hyperparam_in_statedict_new_pl

import torch
import numpy as np
from skimage.metrics import structural_similarity as ssim
import skimage.io as io

import sunpy.visualization.colormaps as cm

### Set environment variable and path to the image to be processed

<font size="3">Configure the I2IWFILM_PATH environment variable to the repository root.</font>

In [ ]:
# get the current directory
current_dir = Path.cwd()
# get the parent directory to get the root directory of the I2IwFiLM repository
parent_dir = current_dir.parent

# set env variable for the dataset path
os.environ["I2IWFILM_PATH"] = str(parent_dir)
print(os.environ["I2IWFILM_PATH"])

# Define and load model

In [ ]:
run_dir = Path(os.environ["I2IWFILM_PATH"] + '/'+ "outputs/wl2cal/S2/I2IwFiLM_S2_whitelight2calcium")
# s1_run_dir = Path(os.environ["I2IWFILM_PATH"] + '/'+ "outputs/wl2cal/S1/f1_I2IwFiLM_S1_seed_0")
print(run_dir)

<font size="3">Load the pretrained translation model from the specified checkpoint path and obtain the inference dataloader.</font>

In [ ]:
config, model, dm, _ = load_from_dir_without_hyperparam_in_statedict_new_pl(
                                    run_path= run_dir.resolve(),
                                    model_path= 'last.ckpt',
                                    load_trainer=False,
                                    override = [
                                    '~callbacks.LR_monitor',
                                    '~callbacks.wandb',
                                    'logger=[]',
                                    #f"model.s1_run_dir={s1_run_dir}",
                                    "++module.load_S1_weights=false",
                                    ]
                                )

test_dataloader = dm.test_dataloader()

# Create batch(es) from new image

First, let's define a few functions that will cut large images into patches, and do/undo image normalization.

In [ ]:
def get_grid_size(full_img, patch_side):
    side = full_img.shape[0] // patch_side
    return side*side

def get_central_crop(img, height, width):
    # Get the center crop
    c, h, w  = img.shape
    assert (h >= height) and (w >= width)

    top = (h - height) // 2
    left = (w - width) // 2
    bottom = top + height
    right = left + width

    img = img[:, top:bottom, left:right]

    return img

def crop_patch(img, patch_index, patch_side, grid_size):
    i = patch_index // int(np.sqrt(grid_size))
    j = patch_index % int(np.sqrt(grid_size))
    return img[
                        i*patch_side: (i+1)*patch_side,
                        j*patch_side: (j+1)*patch_side
                    ].copy()

def do_CenteredNormalScalingKeys(image):
    image = image.copy()
    scaled_image = (image/127.5 - 1.0).astype(np.float32)
    # clip to -1,1
    scaled_image = np.clip(scaled_image, -1.0, 1.0)

    return scaled_image
    
def undo_CenteredNormalScaling(image):
    # 1) rescale to [0, 1]
    image = (image + 1.) / 2.

    # 2) rescale to [0, 255]
    image = image * 255.
    
    # 3) convert to numpy array and to uint8
    image_png = image.permute(1, 2, 0).numpy().astype(np.uint8)
    image_png = np.squeeze(image_png)

    image_npy = image.permute(1, 2, 0).numpy().astype(np.float32)
    image_npy = np.squeeze(image_npy)


    return image_png, image_npy

Define the source image we want to translate and the size of patches provided to the model.

In [ ]:
path = Path(os.environ["I2IWFILM_PATH"] + '/'+ "datasets/WL2CAL/test/temporal_aligned/whitelight/000397_001.npz")
print(path)
image = np.load(path)["arr_0"]
image = image.astype(np.float32)
print(image.shape)

patch_side = test_dataloader.dataset.input_resolution
print(patch_side)
grid_size = get_grid_size(image, patch_side)
print(grid_size)

From the large imafe, create a batch of images to provide to the translation model.

In [ ]:
#batch_v1 = {}
#sample = {}
#idx_patch = 0
#print(image.shape)
#
#if len(image.shape) == 2:
#    image = np.expand_dims(image, axis=0)
#    print(image.shape)
#    sample['cond_image'] = get_central_crop(do_CenteredNormalScalingKeys(image), patch_side, patch_side )
#    sample['cond_path'] = os.path.basename(path)
#
#batch_v1[idx_patch] = sample
#print(batch_v1)

batch_v1 = {}
for index in tqdm(range(grid_size)):
    sample = {} 
    
    idx_img = index // grid_size
    idx_patch = index % grid_size

    tmp_img = crop_patch(image, idx_patch, patch_side, grid_size)
    if len(tmp_img.shape) == 2:
        tmp_img = np.expand_dims(tmp_img, axis=0)

    sample['cond_image'] = do_CenteredNormalScalingKeys(tmp_img)
    sample['cond_path'] = os.path.basename(path)
    
    batch_v1[idx_patch] = sample

#keys_to_tensor = ['cond_image', 'gt_image']
keys_to_tensor = ['cond_image']

batch_v2 = {key: torch.tensor(np.concatenate([d[key][None,:,:] for d in batch_v1.values()])) 
            if key in keys_to_tensor else [d[key] for d in batch_v1.values()] 
            for key in batch_v1[0].keys()}

# Predict batches

Call I2IwFiLM translation on the batch we hust created.

In [ ]:
with torch.no_grad():
    output= model.predict(batch_v2)

From the piece-wise translation, reconstruct the full-size image and write it on disk an the desired location.

In [ ]:
def reconstruct_image(grid_size, batch, outputs, output_dir: Path):
    reconstruction = deepcopy(outputs["Reconstruction"])
    reconstruction = reconstruction.cpu()  
    cur_batch_size = outputs["Reconstruction"].shape[0]

    assert cur_batch_size == grid_size
    
    name = (outputs["Image_name"][0])

    side = int(reconstruction[0].shape[-1] * np.sqrt(grid_size))
    dump = np.zeros((side,side))
    for j, patch in enumerate(reconstruction):
        patch_png, patch = undo_CenteredNormalScaling(patch)
        a = j // int(np.sqrt(grid_size))
        b = j % int(np.sqrt(grid_size))
        dump[
                a*patch.shape[0]: (a+1)*patch.shape[0],
                b*patch.shape[1]: (b+1)*patch.shape[1]
            ] = patch
    out_file = os.path.join(output_dir, f"{name}_sample.npz")
    np.savez_compressed(out_file, dump)

    return out_file

output_dir = Path('./outputs')
out_filename = reconstruct_image(grid_size, batch_v2, output['outputs'], output_dir)


#################################
#output_dir = Path('.')
#out_filename = os.path.basename(path)
#patch_png, patch = undo_CenteredNormalScaling(output['outputs']['Reconstruction'][0].cpu())

#np.savez_compressed(out_filename, patch)

# Display results

Show the initial image ans the results that were written on disk.

In [ ]:
print(path)
print(out_filename)


fig, ax = plt.subplots(1, 2, figsize=(12,4))

sample = np.load(out_filename)['arr_0']
sample_name = os.path.basename(out_filename)
#source = get_central_crop(np.load(path)['arr_0'][None,:,:], patch_side, patch_side)[0]
source = np.load(path)['arr_0']
source_name = os.path.basename(path)

print(source.shape)
print(sample.shape)


sample_name = os.path.basename(out_filename)
sample_name = sample_name.split('.')[0]
source_name = os.path.basename(path)
source_name = source_name.split('.')[0]

ax[0].clear()
ax[1].clear()

cmap_304 = matplotlib.colormaps['sdoaia304']

ax[0].imshow(source, cmap = "gray", interpolation=None, vmin=0, vmax=255)
ax[0].set_title(f'Input image: \n{source_name}')
ax[1].imshow(sample, cmap = "gray", interpolation=None, vmin=0, vmax=255)
ax[1].set_title(f'Translated image: \n{sample_name}')